# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice:Signal analysis and ranking comparison. I chose signal analysis because my goal is to rank content pages by review priority using search performance signals such as impressions, clicks, CTR and average position. This approach fits my lane because it focuses directly on identifying which pages should be reviewed first. I will compare the new ranking with my week 4 baseline on the same validation period and using the same metric.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: time-aware split . I will train the model on earlier March dates and validate it on later March dates. This is more honest than a random split because the model should learn from past data and be evaluated on data that comes later. The same validation period will also be used to evaluate the week 4 baseline so the comparison is fair.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

march_path = hf_hub_download(
    repo_id="Flyrank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

df = pd.read_parquet(march_path)

print("Rows loaded:", len(df))
print("date range:", df["report_date"].min(), "to", df["report_date"].max())

Rows loaded: 9841378
date range: 2026-03-01 to 2026-03-31


In [ ]:
df["report_date"] = pd.to_datetime(df["report_date"])
train_df = df[df["report_date"] <= "2026-03-24"].copy()

val_df = df[df["report_date"] > "2026-03-24"].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Train dates:", train_df["report_date"].min(), "to", train_df["report_date"].max())
print("Validation dates", val_df["report_date"].min(), "to", val_df["report_date"].max())


Train rows: 7548489
Validation rows: 2292889
Train dates: 2026-03-01 00:00:00 to 2026-03-24 00:00:00
Validation dates 2026-03-25 00:00:00 to 2026-03-31 00:00:00


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
val_summary = (
    val_df
    .groupby("content_hash_id", as_index=False)
    .agg({
        "gsc_impressions": "sum",
        "gsc_clicks": "sum",
        "gsc_avg_position": "mean"
    })
)

val_summary["ctr"] = (
    val_summary["gsc_clicks"] /
    val_summary["gsc_impressions"].replace(0, pd.NA)
)

print(val_summary.head())
print("Rows", len(val_summary))

            content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position  \
0  content_000005d4ced12088               30           0         76.166667   
1  content_00001e488b74b799                0           0               NaN   
2  content_00007bd2985b77c3               14           0          3.833333   
3  content_00008950670cb6b5                0           0               NaN   
4  content_0000a348850eb1fc                0           0               NaN   

    ctr  
0   0.0  
1  <NA>  
2   0.0  
3  <NA>  
4  <NA>  
Rows 331436


In [ ]:
ranking_df = val_summary.copy()

ranking_df["ctr"] = ranking_df["ctr"].fillna(0)
ranking_df["gsc_avg_position"] = ranking_df["gsc_avg_position"].fillna(100)

ranking_df["priority_score"] = (
    ranking_df["gsc_impressions"] *
    (1 - ranking_df["ctr"])  *
    ranking_df["gsc_avg_position"]
)

ranking_df = ranking_df.sort_values(
    "priority_score",
    ascending=False
)

print(ranking_df.head(10))

                 content_hash_id  gsc_impressions  gsc_clicks  \
149441  content_73aa61dcedebbf30            27069           2   
70657   content_36e53e9c707674fc            39542          58   
131959  content_66288edeb93b7c4f            79987         422   
131885  content_661a7734f691bef5            39549          11   
169222  content_82e35c4845e6c391            34717          19   
82065   content_3f9e8f387f3fe7e7            19619          12   
324200  content_fa84f5976d5fe3c1            20912          29   
211749  content_a3a1317f7c2bc3dd            21995           1   
221913  content_ab91e088440ace78            18444           0   
25144   content_136c4bf04b07b778            16130          12   

        gsc_avg_position       ctr  priority_score  
149441         49.217629  0.000074    1.332174e+06  
70657          31.773706  0.001467    1.254553e+06  
131959         13.738387  0.005276    1.093095e+06  
131885         26.531121  0.000278    1.048987e+06  
169222         30.1

/tmp/ipykernel_2424/4030547879.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ranking_df["ctr"] = ranking_df["ctr"].fillna(0)


In [33]:
ranking_df = ranking_df.reset_index(drop=True)

ranking_df["new_rank"] = range(1, len(ranking_df)+ 1)

print(
    ranking_df[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "priority_score",
            "new_rank"
        ]
    ].head(10)
)

            content_hash_id  gsc_impressions  gsc_clicks       ctr  \
0  content_73aa61dcedebbf30            27069           2  0.000074   
1  content_36e53e9c707674fc            39542          58  0.001467   
2  content_66288edeb93b7c4f            79987         422  0.005276   
3  content_661a7734f691bef5            39549          11  0.000278   
4  content_82e35c4845e6c391            34717          19  0.000547   
5  content_3f9e8f387f3fe7e7            19619          12  0.000612   
6  content_fa84f5976d5fe3c1            20912          29  0.001387   
7  content_a3a1317f7c2bc3dd            21995           1  0.000045   
8  content_ab91e088440ace78            18444           0  0.000000   
9  content_136c4bf04b07b778            16130          12  0.000744   

   gsc_avg_position  priority_score  new_rank  
0         49.217629    1.332174e+06         1  
1         31.773706    1.254553e+06         2  
2         13.738387    1.093095e+06         3  
3         26.531121    1.048987e+06  

In [37]:
import numpy as np

baseline_df = val_summary.copy()

baseline_df["ctr"] = (
    baseline_df["gsc_clicks"] /
    baseline_df["gsc_impressions"].replace(0, pd.NA)
).fillna(0)

baseline_df["expected_ctr"] = np.select(
    [
        baseline_df["gsc_avg_position"] <= 3,
        baseline_df["gsc_avg_position"] <= 10,
        baseline_df["gsc_avg_position"] <= 20
    ],
    [0.004576, 0.003473, 0.002770],
    default=0.001289
)

baseline_df["baseline_score"] = (
    baseline_df["gsc_impressions"]*
    (baseline_df["expected_ctr"] - baseline_df["ctr"]).clip(lower=0)
)

baseline_df = baseline_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_df["baseline_rank"] = range(1, len(baseline_df) + 1)

print(
    baseline_df[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "expected_ctr",
            "baseline_score",
            "baseline_rank"
        ]
    ].head(10)
)




            content_hash_id  gsc_impressions  gsc_clicks       ctr  \
0  content_44f34c0a90047651           171954           7  0.000041   
1  content_fec55986a1868d62            92776           0  0.000000   
2  content_0e03de7680314cd5           139835         244  0.001745   
3  content_8e1334d6356668e3            70161           0  0.000000   
4  content_8d7d99f109e19aa2            63787          24  0.000376   
5  content_545bb6cc7081ded3            55403          43  0.000776   
6  content_046fc480045b88f5            54713           4  0.000073   
7  content_9ef3d7516483e665            46778          37  0.000791   
8  content_4ffe18112a5642e3            69580         148  0.002127   
9  content_bf078007df823490            43649           0  0.000000   

   gsc_avg_position  expected_ctr  baseline_score  baseline_rank  
0          2.175926      0.004576      779.861504              1  
1          0.735728      0.004576      424.542976              2  
2          2.421711      0.0

/tmp/ipykernel_2424/129858294.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [38]:
comparison_df = baseline_df[
    ["content_hash_id", "baseline_rank", "baseline_score"]
].merge(
    ranking_df[
        ["content_hash_id", "new_rank", "priority_score"]
    ],
    on="content_hash_id",
    how="inner"
)

comparison_df["rank-change"] = (
    comparison_df["baseline_rank"] - comparison_df["new_rank"]
)

comparison_df = comparison_df.sort_values(
    "new_rank"
)

print(comparison_df.head(10))


                 content_hash_id  baseline_rank  baseline_score  new_rank  \
135     content_73aa61dcedebbf30            136       32.891941         1   
262144  content_36e53e9c707674fc         262145        0.000000         2   
224221  content_66288edeb93b7c4f         224222        0.000000         3   
85      content_661a7734f691bef5             86       39.978661         4   
248     content_82e35c4845e6c391            249       25.750213         5   
921     content_3f9e8f387f3fe7e7            922       13.288891         6   
161839  content_fa84f5976d5fe3c1         161840        0.000000         7   
217     content_a3a1317f7c2bc3dd            218       27.351555         8   
299     content_ab91e088440ace78            300       23.774316         9   
1837    content_136c4bf04b07b778           1838        8.791570        10   

        priority_score  rank-change  
135       1.332174e+06          135  
262144    1.254553e+06       262143  
224221    1.093095e+06       224219  


In [ ]:
comparison_table = comparison_df[
    [
        "content_hash_id",
        "baseline_rank",
        "new_rank",
        "rank-change",
        "priority_score"
    ]
].head(10)

comparison_table

,content_hash_id,baseline_rank,new_rank,rank-change,priority_score
53,content_73aa61dcedebbf30,54,1,53,1.332174e+06
25,content_36e53e9c707674fc,26,2,24,1.254553e+06
4,content_66288edeb93b7c4f,5,3,2,1.093095e+06
24,content_661a7734f691bef5,25,4,21,1.048987e+06
33,content_82e35c4845e6c391,34,5,29,1.047175e+06
128,content_3f9e8f387f3fe7e7,129,6,123,9.753288e+05
108,content_fa84f5976d5fe3c1,109,7,102,8.189318e+05
94,content_a3a1317f7c2bc3dd,95,8,87,7.945984e+05
146,content_ab91e088440ace78,147,9,138,7.704454e+05
197,content_136c4bf04b07b778,198,10,188,7.686408e+05


The new ranking method changes the review priority significally compared with the baseline. Some pages that ranked much lower using impressions alone moved to the top when CTR and average search position were included. This suggests that using multiple search performance signals gives a more useful review priority than relying only on impressions.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
comparison_df.sort_values(
    "rank-change",
    ascending=False
).head(10)

,content_hash_id,baseline_rank,new_rank,rank-change,priority_score
197,content_136c4bf04b07b778,198,10,188,7.686408e+05
146,content_ab91e088440ace78,147,9,138,7.704454e+05
128,content_3f9e8f387f3fe7e7,129,6,123,9.753288e+05
108,content_fa84f5976d5fe3c1,109,7,102,8.189318e+05
94,content_a3a1317f7c2bc3dd,95,8,87,7.945984e+05
53,content_73aa61dcedebbf30,54,1,53,1.332174e+06
33,content_82e35c4845e6c391,34,5,29,1.047175e+06
25,content_36e53e9c707674fc,26,2,24,1.254553e+06
24,content_661a7734f691bef5,25,4,21,1.048987e+06
4,content_66288edeb93b7c4f,5,3,2,1.093095e+06


The largest ranking changes show that the new method can strongly re priotitize some pages. For example some pages moved from much lower positions in the baseline to the top of the new ranking. This happens because the new score uses CTR and average search position in addition to impressions. A limitation is that the priority score may over favor pages with poor search position or very low CTR, even if those pages are not the important business pages. It also depends only on search performance signals and does not include content quality, conversions or business value. Because of this the ranking should be used as a review priority signal rather than a final decision.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.